In [18]:
# Install all required libraries
!pip -q install chromadb groq pypdf nltk requests

# Download NLTK resources
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

True

In [19]:
import os
import re
import requests
import nltk
import chromadb
from pypdf import PdfReader
from groq import Groq

nltk.download("punkt")
nltk.download("punkt_tab")

# ---- PASTE YOUR KEYS HERE (or set as environment variables) ----
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')   # only needed if EMBEDDING_PROVIDER = "openai"

HF_TOKEN = os.environ["HF_TOKEN"]
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

PDF_PATH = "./policy.pdf"          # <-- put your PDF here
CHROMA_DIR = "./chromadb"         # local persistent folder for ChromaDB
COLLECTION_NAME = "health_policy"

# ---- EMBEDDING PROVIDER SWITCH ----
EMBEDDING_PROVIDER = "huggingface"   # "huggingface" (free) or "openai" (paid)
# EMBEDDING_PROVIDER = "openai"   # "huggingface" (free) or "openai" (paid)

HF_EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
HF_EMBED_URL = f"https://router.huggingface.co/hf-inference/models/{HF_EMBED_MODEL}/pipeline/feature-extraction"

OPENAI_EMBED_MODEL = "text-embedding-3-small"
OPENAI_EMBED_URL = "https://api.openai.com/v1/embeddings"

GEN_MODEL = "llama-3.3-70b-versatile"
print("All libraries imported successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


All libraries imported successfully!


In [20]:
#upload the required PDF  file
from google.colab import files

uploaded = files.upload()

Saving policy.pdf to policy (1).pdf


In [21]:
#Verify if PDF file exists
PDF_PATH = "policy.pdf"

if os.path.exists(PDF_PATH):
    print("✅ PDF uploaded successfully.")
else:
    print("❌ PDF not found.")

✅ PDF uploaded successfully.


In [22]:
from pypdf import PdfReader

PDF_PATH = "policy.pdf"      # Replace with your PDF file name

reader = PdfReader(PDF_PATH)

print("Number of pages:", len(reader.pages))

Number of pages: 69


In [23]:
def looks_like_heading(line):

    line = line.strip()

    if not line or len(line) > 60:
      print(f"'{line}' --> False")
      return False

    if line.isupper():
      print(f"'{line}' --> True")
      return True

    # Numbered headings like 1.1 Standard Definitions
    if re.match(r'^\d+(\.\d+)*\.?\s+[A-Z]', line):
        print(f"'{line}' --> True")
        return True

    # SECTION A. DEFINITIONS
    if line.upper().startswith("SECTION"):
        print(f"'{line}' --> True")
        return True

    # Single capitalized word
    if len(line.split()) == 1 and line[0].isupper():
        print(f"'{line}' --> True")
        return True

    words = line.split()

    # Single word heading
    if len(words) == 1 and words[0][0].isupper():
        print(f"'{line}' --> True")
        return True

    if len(words) >= 2 and sum(1 for w in words if w[:1].isupper()) / len(words) > 0.7:
        if not line.endswith("."):
            print(f"'{line}' --> True")
            return True

    print(f"'{line}' --> False")
    return False


In [24]:
def read_pdf(PDF_PATH):
    reader = PdfReader(PDF_PATH)
    pages_data = []
    current_section = "General"

    for page_num, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        lines = text.split("\n")

        page_section = current_section
        page_lines = []
        for line in lines:
            if looks_like_heading(line):
                current_section = line.strip()
                page_section = current_section
            page_lines.append(line)

        pages_data.append({
            "page": page_num,
            "section": page_section,
            "text": "\n".join(page_lines).strip()
        })

    return pages_data


pages_data = read_pdf(PDF_PATH)
print(f"Read {len(pages_data)} pages from {PDF_PATH}")
print(pages_data[0] if pages_data else "No pages found")

'HDFC ERGO General Insurance Company Limited' --> True
'' --> False
'Policy Wording' --> True
'my: Optima Secure' --> False
'' --> False
'HDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146 CIN: U66030MH2007PLC177117. Registered & Corporate Office: 6th Floor,' --> False
'Leela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059.' --> False
'UIN: my: Optima Secure - HDFHLIP26058V082526' --> False
'' --> False
'1' --> False
'Table of Contents' --> False
'Sr. No. Particulars Page No.' --> False
'Preamble 2' --> False
'Operative Clause 2' --> False
'A.1.1 Standard Definitions 2' --> True
'A.1.2 Specific Definitions 2' --> True
'B.1 Base Coverages 11' --> True
'B.2 Optional Coverages 14' --> True
'B.3 Renewal Benefit 27' --> True
'C.1 Waiting Periods 30' --> True
'C.2 Standard Exclusions 31' --> True
'C.3 Specific Exclusions 33' --> True
'D.1 Standard General Terms & Clauses  34' --> True
'E Other Terms & Clauses 48' --> False
'Annexure A 52' --> False
'Annexure B

In [25]:
#====================================================================================================================================================================
#============================================================== Semantic Chunking using NTLK ========================================================================
#====================================================================================================================================================================

from nltk.tokenize import sent_tokenize

CHUNK_TARGET_CHARS = 500   # rough target size per chunk
OVERLAP_SENTENCES = 2      # how many trailing sentences to carry into the next chunk

def chunk_pages(pages_data, target_chars=CHUNK_TARGET_CHARS, overlap_sentences=OVERLAP_SENTENCES):
    chunks = []

    for page_info in pages_data:
        text = page_info["text"]
        if not text:
            continue

        sentences = sent_tokenize(text)

        current_chunk = []
        current_len = 0

        for sentence in sentences:
            current_chunk.append(sentence)
            current_len += len(sentence)

            if current_len >= target_chars:
                chunks.append({
                    "text": " ".join(current_chunk),
                    "page": page_info["page"],
                    "section": page_info["section"]
                })
                # start the next chunk with the last few sentences of this one (overlap)
                current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences else []
                current_len = sum(len(s) for s in current_chunk)

        # leftover sentences on this page (only keep if it has content beyond the carried-over overlap)
        if current_chunk and len(current_chunk) > overlap_sentences:
            chunks.append({
                "text": " ".join(current_chunk),
                "page": page_info["page"],
                "section": page_info["section"]
            })

    return chunks


chunks = chunk_pages(pages_data)
print(f"Created {len(chunks)} chunks")
print(chunks[0] if chunks else "No chunks created")
for i, chunk in enumerate(chunks[:5], start=1):
    print("="*80)
    print(f"Chunk {i}")
    print(f"Page    : {chunk['page']}")
    print(f"Section : {chunk['section']}")
    print(f"Length  : {len(chunk['text'])} characters")
    print("-"*80)
    print(chunk["text"])
    print()

Created 474 chunks
{'text': 'HDFC ERGO General Insurance Company Limited  \n \nPolicy Wording \nmy: Optima Secure  \n \nHDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146 CIN: U66030MH2007PLC177117. Registered & Corporate Office: 6th Floor, \nLeela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059. UIN: my: Optima Secure - HDFHLIP26058V082526 \n \n1 \nTable of Contents \nSr. No. Particulars Page No. Preamble 2 \nOperative Clause 2 \nA.1.1 Standard Definitions 2 \nA.1.2 Specific Definitions 2 \nB.1 Base Coverages 11 \nB.2 Optional Coverages 14 \nB.3 Renewal Benefit 27 \nC.1 Waiting Periods 30 \nC.2 Standard Exclusions 31 \nC.3 Specific Exclusions 33 \nD.1 Standard General Terms & Clauses  34 \nE Other Terms & Clauses 48 \nAnnexure A 52 \nAnnexure B 56 \nAnnexure C 60', 'page': 1, 'section': 'D.1 Standard General Terms & Clauses  34'}
Chunk 1
Page    : 1
Section : D.1 Standard General Terms & Clauses  34
Length  : 753 characters
-----------------------------

In [26]:
#====================================================================================================================================================================
#============================================================== Create Embeddings ===================================================================================
#====================================================================================================================================================================

def get_embedding_hf(text):
    response = requests.post(
        HF_EMBED_URL,
        headers={"Authorization": f"Bearer {HF_TOKEN}"},
        json={"inputs": text, "options": {"wait_for_model": True}},
        timeout=60
    )
    response.raise_for_status()
    result = response.json()

    if isinstance(result, dict) and "error" in result:
        raise RuntimeError(f"HF Inference API error: {result['error']}")

    vector = result
    if isinstance(vector[0], list) and isinstance(vector[0][0], list):
        import numpy as np
        vector = np.mean(np.array(vector[0]), axis=0).tolist()
    elif isinstance(vector[0], list):
        vector = vector[0] if isinstance(vector[0][0], float) else vector
    return vector


def get_embedding_openai(text):
    response = requests.post(
        OPENAI_EMBED_URL,
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
        json={"input": text, "model": OPENAI_EMBED_MODEL}
    )
    response.raise_for_status()
    result = response.json()
    return result["data"][0]["embedding"]


def get_embedding(text):
    if EMBEDDING_PROVIDER == "openai":
        return get_embedding_openai(text)
    elif EMBEDDING_PROVIDER == "huggingface":
        return get_embedding_hf(text)
    else:
        raise ValueError(f"Unknown EMBEDDING_PROVIDER: {EMBEDDING_PROVIDER}")

input_text = "This is a test sentence."
test_vec = get_embedding(input_text)
print(f"Using provider: {EMBEDDING_PROVIDER}")
print(f"Input Text: {input_text}")
print("Calling Embedding API...Done")
print(f"Embedding Length: {len(test_vec)}")
print("First 10 Values:", test_vec[:10])

Using provider: huggingface
Input Text: This is a test sentence.
Calling Embedding API...Done
Embedding Length: 384
First 10 Values: [0.08429645001888275, 0.05795371159911156, 0.00449335528537631, 0.10582107305526733, 0.0070834094658494, -0.01784464716911316, -0.01688799448311329, -0.015228280797600746, 0.04047312214970589, 0.03342258557677269]


In [27]:
#====================================================================================================================================================================
#============================================================== Store chunks + embeddings + metadata in ChromaDB ====================================================
#====================================================================================================================================================================

# Create/Open persistent database
print("=" * 80)
print("Creating/Opening ChromaDB Persistent Client...")
print(f"Database Path : {CHROMA_DIR}")

# Create/Open persistent database
client = chromadb.PersistentClient(path=CHROMA_DIR)

print("Persistent Client Created Successfully.\n")

# Check existing collections
print("=" * 80)
print("Fetching existing collections...")

# Check whether collection already exists
existing_collections = [c.name for c in client.list_collections()]

print("Existing Collections :", existing_collections)
print()

# Check if collection exists
if COLLECTION_NAME in existing_collections:
    print(f"Using existing collection: {COLLECTION_NAME}")
    collection = client.get_collection(COLLECTION_NAME)
else:
    print(f"Creating new collection: {COLLECTION_NAME}")
    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}
    )

    print("Collection Created Successfully.")
    print()

    ids, embeddings, documents, metadatas = [], [], [], []
    print("Generating Embeddings and Preparing Data...\n")

    for i, chunk in enumerate(chunks):
        emb = get_embedding(chunk["text"])
        ids.append(f"chunk_{i}")
        embeddings.append(emb)
        documents.append(chunk["text"])
        metadatas.append({"page": chunk["page"], "section": chunk["section"]})

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )

print(f"Stored {collection.count()} chunks in ChromaDB at '{CHROMA_DIR}'")

Creating/Opening ChromaDB Persistent Client...
Database Path : ./chromadb
Persistent Client Created Successfully.

Fetching existing collections...
Existing Collections : ['health_policy']

Using existing collection: health_policy
Stored 474 chunks in ChromaDB at './chromadb'


In [28]:
#====================================================================================================================================================================
#============================================================== Top-K Retrieval (Cosine Similarity Matrix) ==========================================================
#====================================================================================================================================================================


def retrieve_top_k(query, k=3):
    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    retrieved = []
    for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
        retrieved.append({
            "text": doc,
            "section": meta["section"],
            "page": meta["page"],
            "distance": dist
        })
    return retrieved


sample_results = retrieve_top_k("What is the waiting period?", k=3)
for i, r in enumerate(sample_results, start=1):
    print(f"Occurrence {i}")
    print(r["text"])
    print("-" * 80)

Occurrence 1
Waiting Period  means a period from the inception of this Policy during which specified 
diseases/treatments are not covered. On completion of the Waiting Period, diseases/treatments 
shall be covered provided the Policy has been continuously renewed without any break. SECTION B. BENEFITS 
1. Base Coverage 
The Covers listed below are in-built Policy benefits and shall be available to all Insured Persons in 
accordance with the procedures set out in this Policy and up to the Sub -limits mentioned in the 
Policy Schedule.
--------------------------------------------------------------------------------
Occurrence 2
c. Loadings shall be applied from Commencement Date including subsequent 
Renewal(s), and on increased Sum Insured. d. Proposer shall be informed about the proposed loading with premium, specific 
Waiting Period or permanent exclusion (if any) through a counter offer letter and 
Policy will be issued only on specific acceptance within 15 days of the receipt of suc

In [31]:
#====================================================================================================================================================================
#============================================================== Generation (Groq - Llama 3.3 70B versatile) =========================================================
#====================================================================================================================================================================

groq_client = Groq(api_key=GROQ_API_KEY)

PROMPT_TEMPLATE = """
You are a health insurance policy assistant.
Answer ONLY using the provided context.
If the answer isn't in the context, say so — do not guess.

Context: {retrieved_chunks_with_section_path_and_page}
Question:
{user_query}

Answer, and cite the section and page for each claim,
e.g. (Source: Waiting Periods, p.12).
"""


def format_context(retrieved_chunks):
    print("\n========== format_context() ==========\n")

    formatted = []

    #for i, r in enumerate(retrieved_chunks, start=1):

        #print(f"Chunk {i}")
        #print("Section :", r["section"])
        #print("Page    :", r["page"])
        #print("Text    :")
        #print(r["text"])
        #print("-" * 80)#

    formatted.append(f"[Section: {r['section']}, Page: {r['page']}]\n{r['text']}")

    context = "\n\n".join(formatted)

    print("\n========== Final Context ==========\n")
    print(context)

    return context
    #return "\n\n".join(formatted)


def generate_answer(user_query, k=3):

    retrieved_chunks = retrieve_top_k(user_query, k=k)

    context = format_context(retrieved_chunks)

    print("\n===================================================")
    print("Step 3 : Creating Prompt")
    print("===================================================")

    prompt = PROMPT_TEMPLATE.format(
        retrieved_chunks_with_section_path_and_page=context,
        user_query=user_query
    )

    print(prompt)

    print("\n===================================================")
    print("Step 4 : Calling Groq LLM")
    print("===================================================")

    response = groq_client.chat.completions.create(
        model=GEN_MODEL,
        messages=[
            {
             "role": "user",
             "content": prompt
            }
        ],
        temperature=0.2
    )

    print("\nGroq Response Received Successfully")

    print("\n===================================================")
    print("Step 5 : Generated Answer")
    print("===================================================")


    print(textwrap.fill(response.choices[0].message.content, width=120))

    return response.choices[0].message.content, retrieved_chunks

In [32]:
#====================================================================================================================================================================
#============================================================== Ask a Question ======================================================================================
#====================================================================================================================================================================
import textwrap
# question = "What is the sum insured for Room Rent limits?"
while True:
    question = input("Ask me anything about the health insurance policy. ").strip().lower()
    answer, used_chunks = generate_answer(question, k=3)

    # print("QUESTION:", question)
    print("\nANSWER:\n")
    wrapped_text = textwrap.fill(answer, width=120)
    print(wrapped_text)

    # Ask if they want to continue
    quit_choice = input("Do you want to continue? (yes/no): ").strip().lower()
    if quit_choice == "no":
        print("Goodbye!")
        break

Ask me anything about the health insurance policy. what is the waiting period?

========== format_context() ==========


========== Final Context ==========

[Section: Surgical Procedures, Page: 31]
iii. If any of the specified disease/procedure falls under the waiting period specified for Pre -
Existing diseases, then the longer of the two waiting periods shall apply. iv. The waiting period for listed conditions shall apply even if contracted after the Policy or 
declared and accepted without a specific exclusion. v. If the Insured Person is continuously covered without any break as defined under the 
applicable norms on portability stipulated by IRDAI, then waiting period for the same would 
be reduced to the extent of prior coverage.

Step 3 : Creating Prompt

You are a health insurance policy assistant. 
Answer ONLY using the provided context. 
If the answer isn't in the context, say so — do not guess.

Context: [Section: Surgical Procedures, Page: 31]
iii. If any of the specified 